# Simulador de Sincronización Discreta tipo Kuramoto sobre Grafos

**Modelo:** $\left(F_\kappa(\theta)\right)_i = \theta_i + \omega^* + \sigma_\kappa\left(S_i(\theta)\right) \mod M$

donde $S_i(\theta) = \sum_{j \in N(i)} d_M^{\text{signed}}(\theta_j, \theta_i)$ y $\sigma_\kappa(s) = +1$ si $s > \kappa$, $-1$ si $s < -\kappa$, $0$ si $|s| \leq \kappa$.

In [1]:
%matplotlib inline

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output
from itertools import product
from math import comb

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 1. Funciones auxiliares

In [2]:
def bell_number(n):
    """
    Calcula el número de Bell B_n mediante programación dinámica.
    B_n cuenta el número de particiones de un conjunto con n elementos.
    """
    bell = [[0 for _ in range(n + 1)] for _ in range(n + 1)]
    bell[0][0] = 1
    for i in range(1, n + 1):
        bell[i][0] = bell[i - 1][i - 1]
        for j in range(1, i + 1):
            bell[i][j] = bell[i - 1][j - 1] + bell[i][j - 1]
    return bell[n][0]


def kuramoto_order_parameter(theta, M):
    """
    Parámetro de orden discreto tipo Kuramoto.

    r(t) = |(1/N) sum_j exp(2 pi i theta_j / M)|

    r = 1 indica sincronización perfecta.
    r cercano a 0 indica dispersión de fases.
    """
    theta = np.array(theta)
    z = np.exp(2j * np.pi * theta / M)
    return np.abs(np.mean(z))


print(f"Bell(3) = {bell_number(3)}, Bell(4) = {bell_number(4)}, Bell(5) = {bell_number(5)}")

Bell(3) = 5, Bell(4) = 15, Bell(5) = 52


## 2. Clase principal del modelo

In [3]:
class KuramotoDiscreto:
    def __init__(self, N=4, M=6, kappa=2, omega=0, tipo_grafo="Completo", epsilon=0):
        self.N = int(N)
        self.M = int(M)
        self.kappa = int(kappa)
        self.omega = int(omega)
        self.tipo_grafo = tipo_grafo
        self.epsilon = int(epsilon)
        self.G = self.crear_grafo()
        self.pos = nx.spring_layout(self.G, seed=7)

    def crear_grafo(self):
        if self.tipo_grafo == "Completo":
            return nx.complete_graph(self.N)
        elif self.tipo_grafo == "Camino":
            return nx.path_graph(self.N)
        elif self.tipo_grafo == "Ciclo":
            return nx.cycle_graph(self.N)
        elif self.tipo_grafo == "Estrella":
            return nx.star_graph(self.N - 1)
        else:
            raise ValueError(f"Tipo de grafo no reconocido: {self.tipo_grafo}")

    def dM_signed(self, a, b):
        """
        Diferencia circular con signo en Z_M.
        Mide cuánto 'jala' b hacia a, con signo.

        d_M^signed(a, b) = ((b - a + floor(M/2)) mod M) - floor(M/2)

        Positivo: b está 'adelante' de a en Z_M.
        Negativo: b está 'atrás' de a.
        """
        return ((int(b) - int(a) + self.M // 2) % self.M) - self.M // 2

    def dM_circular(self, a, b):
        """
        Distancia circular sin signo:
        d_M(a, b) = min(|a-b|, M - |a-b|)
        """
        diff = abs(int(a) - int(b))
        return min(diff, self.M - diff)

    def sigma_kappa(self, s):
        """
        Función signo umbralizada:
        sigma_kappa(s) = +1 si s > kappa
                       = -1 si s < -kappa
                       =  0 si |s| <= kappa
        """
        if s > self.kappa:
            return 1
        elif s < -self.kappa:
            return -1
        else:
            return 0

    def S_i(self, theta, i):
        """
        Suma de acoplamiento con signo del vértice i:
        S_i(theta) = sum_{j in N(i)} d_M^signed(theta_i, theta_j)

        Positivo: los vecinos en promedio están 'adelante' de i.
        Negativo: los vecinos en promedio están 'atrás' de i.
        """
        return sum(self.dM_signed(theta[i], theta[j]) for j in self.G.neighbors(i))

    def paso(self, theta):
        """
        Aplica una iteración de F_kappa a la configuración theta.
        Actualización simultánea de todos los vértices.
        """
        theta = np.array(theta, dtype=int)
        nueva = np.zeros_like(theta)
        for i in range(self.N):
            s_i = self.S_i(theta, i)
            nueva[i] = (theta[i] + self.omega + self.sigma_kappa(s_i)) % self.M
        return nueva

    def subred_acuerdo_edges(self, theta):
        """
        Devuelve las aristas de la subred epsilon-sincronizada G^epsilon_theta.
        (i, j) pertenece si d_M(theta_i, theta_j) <= epsilon.
        """
        edges = []
        for i, j in self.G.edges():
            if self.dM_circular(theta[i], theta[j]) <= self.epsilon:
                edges.append((i, j))
        return tuple(sorted(edges))

    def esta_sincronizado(self, theta):
        """
        Verifica si theta está en la diagonal Delta_M.
        """
        theta = np.array(theta)
        return bool(np.all(theta == theta[0]))

    def simular(self, theta0, T=30):
        """
        Simula una trayectoria hasta T pasos.
        """
        theta = np.array(theta0, dtype=int)
        trayectoria = [theta.copy()]
        S_hist = []
        sigma_hist = []
        r_hist = [kuramoto_order_parameter(theta, self.M)]
        subredes = [self.subred_acuerdo_edges(theta)]

        for _ in range(T):
            S_t = [self.S_i(theta, i) for i in range(self.N)]
            sigma_t = [self.sigma_kappa(s) for s in S_t]
            S_hist.append(S_t)
            sigma_hist.append(sigma_t)
            theta = self.paso(theta)
            trayectoria.append(theta.copy())
            r_hist.append(kuramoto_order_parameter(theta, self.M))
            subredes.append(self.subred_acuerdo_edges(theta))

        return {
            "trayectoria": np.array(trayectoria),
            "S_hist": np.array(S_hist),
            "sigma_hist": np.array(sigma_hist),
            "r_hist": np.array(r_hist),
            "subredes": subredes,
        }

    def clasificar_atractor(self, theta0, T_max=200):
        """
        Clasifica la trayectoria como:
        - sync
        - punto fijo no trivial
        - ciclo
        - transitorio
        """
        theta = np.array(theta0, dtype=int)
        vistos = {}

        for t in range(T_max + 1):
            key = tuple(theta.tolist())

            if self.esta_sincronizado(theta):
                return {
                    "tipo": "sync",
                    "tiempo": t,
                    "periodo": 1,
                    "estado_final": theta.copy()
                }

            if key in vistos:
                t0 = vistos[key]
                periodo = t - t0
                tipo = "punto fijo no trivial" if periodo == 1 else "ciclo"
                return {
                    "tipo": tipo,
                    "tiempo": t0,
                    "periodo": periodo,
                    "estado_final": theta.copy()
                }

            vistos[key] = t
            theta = self.paso(theta)

        return {
            "tipo": "transitorio",
            "tiempo": T_max,
            "periodo": None,
            "estado_final": theta.copy()
        }

    def tabla_trayectoria(self, resultado):
        """
        Construye una tabla con theta_t, S_i, sigma_kappa(S_i) y r(t).
        """
        trayectoria = resultado["trayectoria"]
        S_hist = resultado["S_hist"]
        sigma_hist = resultado["sigma_hist"]
        r_hist = resultado["r_hist"]

        filas = []
        for t in range(len(trayectoria)):
            fila = {
                "t": t,
                "theta(t)": tuple(trayectoria[t]),
                "r(t)": round(float(r_hist[t]), 4),
                "subred_acuerdo": resultado["subredes"][t]
            }
            if t < len(S_hist):
                fila["S_i(theta_t)"] = tuple(S_hist[t])
                fila["sigma_kappa(S_i)"] = tuple(sigma_hist[t])
            else:
                fila["S_i(theta_t)"] = "-"
                fila["sigma_kappa(S_i)"] = "-"
            filas.append(fila)

        return pd.DataFrame(filas)


print("Clase KuramotoDiscreto cargada correctamente.")

Clase KuramotoDiscreto cargada correctamente.


## 3. Exploración exhaustiva

In [4]:
def exploracion_exhaustiva(N, M, kappa, omega, tipo_grafo, epsilon, T_max=150):
    modelo = KuramotoDiscreto(N=N, M=M, kappa=kappa, omega=omega,
                              tipo_grafo=tipo_grafo, epsilon=epsilon)
    total = M ** N
    conteo = {
        "sync": 0,
        "punto fijo no trivial": 0,
        "ciclo": 0,
        "transitorio": 0
    }
    tiempos_sync = []
    subredes_realizables = set()
    transiciones = set()

    for theta0 in product(range(M), repeat=N):
        clasif = modelo.clasificar_atractor(theta0, T_max=T_max)
        conteo[clasif["tipo"]] += 1

        if clasif["tipo"] == "sync":
            tiempos_sync.append(clasif["tiempo"])

        resultado = modelo.simular(theta0, T=min(T_max, 50))
        subredes = resultado["subredes"]

        for s in subredes:
            subredes_realizables.add(s)
        for a, b in zip(subredes[:-1], subredes[1:]):
            if a != b:
                transiciones.add((a, b))

    resumen = {
        "total_configuraciones": total,
        "sync": conteo["sync"],
        "fix": conteo["punto fijo no trivial"],
        "cyc": conteo["ciclo"],
        "transitorio": conteo["transitorio"],
        "rho_sync": conteo["sync"] / total,
        "rho_fix": conteo["punto fijo no trivial"] / total,
        "rho_cyc": conteo["ciclo"] / total,
        "rho_transitorio": conteo["transitorio"] / total,
        "tiempos_sync": tiempos_sync,
        "num_subredes_realizables": len(subredes_realizables),
        "num_transiciones": len(transiciones),
        "Bell_BN": bell_number(N),
        "subredes_realizables": subredes_realizables,
        "transiciones": transiciones,
    }
    return resumen


print("Función exploracion_exhaustiva cargada.")

Función exploracion_exhaustiva cargada.


## 4. Barrido de kappa: mapa de regímenes

In [5]:
def barrido_kappa(N, M, omega, tipo_grafo, epsilon, T_max=150):
    """
    Barre kappa desde 1 hasta kappa_max = deg_max * floor(M/2)
    y calcula rho_sync, rho_fix, rho_cyc para cada valor.
    """
    modelo_temp = KuramotoDiscreto(N=N, M=M, kappa=1, omega=omega,
                                   tipo_grafo=tipo_grafo, epsilon=epsilon)
    degmax = max(dict(modelo_temp.G.degree()).values())
    kappa_max = degmax * (M // 2)

    filas = []
    for kappa in range(1, kappa_max + 1):
        resumen = exploracion_exhaustiva(N, M, kappa, omega, tipo_grafo, epsilon, T_max=T_max)
        filas.append({
            "kappa": kappa,
            "rho_sync": resumen["rho_sync"],
            "rho_fix": resumen["rho_fix"],
            "rho_cyc": resumen["rho_cyc"],
            "rho_transitorio": resumen["rho_transitorio"],
            "subredes_realizables": resumen["num_subredes_realizables"],
            "transiciones": resumen["num_transiciones"],
        })

    return pd.DataFrame(filas)


print("Función barrido_kappa cargada.")

Función barrido_kappa cargada.


## 5. Visualizaciones

In [6]:
def plot_heatmap(resultado, M):
    trayectoria = resultado["trayectoria"]
    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(trayectoria.T, aspect="auto", interpolation="nearest",
                   cmap="hsv", vmin=0, vmax=M - 1)
    ax.set_xlabel("Tiempo t", fontsize=12)
    ax.set_ylabel("Vértice i", fontsize=12)
    ax.set_title("Heatmap de evolución de fases $\\theta_i(t)$", fontsize=13)
    ax.set_yticks(range(trayectoria.shape[1]))
    ax.set_yticklabels([f"v{i}" for i in range(trayectoria.shape[1])])
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Fase en $\\mathbb{Z}_M$")
    plt.tight_layout()
    plt.show()


def plot_order_parameter(resultado, clasif=None):
    r = resultado["r_hist"]
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(range(len(r)), r, marker="o", markersize=4, linewidth=2, color="steelblue")
    ax.axhline(y=1.0, color="green", linestyle="--", alpha=0.7, label="r=1 (sync)")
    ax.fill_between(range(len(r)), r, alpha=0.15, color="steelblue")
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("Tiempo t", fontsize=12)
    ax.set_ylabel("r(t)", fontsize=12)
    titulo = "Parámetro de orden de Kuramoto r(t)"
    if clasif:
        titulo += f" — Atractor: {clasif['tipo']}"
    ax.set_title(titulo, fontsize=13)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_grafo(modelo, theta, title="Grafo coloreado por fase"):
    fig, ax = plt.subplots(figsize=(5, 5))
    node_colors = np.array(theta, dtype=float)

    # Aristas normales (tenues)
    aristas_normales = [(i, j) for i, j in modelo.G.edges()
                        if modelo.dM_circular(theta[i], theta[j]) > modelo.epsilon]
    nx.draw_networkx_edges(modelo.G, modelo.pos, edgelist=aristas_normales,
                           ax=ax, alpha=0.25, width=1.0)

    # Aristas de acuerdo (resaltadas)
    acuerdo_edges = modelo.subred_acuerdo_edges(theta)
    if acuerdo_edges:
        nx.draw_networkx_edges(modelo.G, modelo.pos, edgelist=list(acuerdo_edges),
                               ax=ax, width=3.5, edge_color="green", alpha=0.8)

    nodes = nx.draw_networkx_nodes(
        modelo.G, modelo.pos,
        node_color=node_colors,
        cmap=plt.cm.hsv,
        vmin=0, vmax=modelo.M - 1,
        node_size=700, ax=ax
    )
    nx.draw_networkx_labels(modelo.G, modelo.pos, ax=ax,
                            labels={i: f"v{i}\n{theta[i]}" for i in range(modelo.N)},
                            font_size=9)
    cbar = plt.colorbar(nodes, ax=ax)
    cbar.set_label("Fase")
    ax.set_title(title, fontsize=12)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def plot_hist_tiempos(tiempos):
    if len(tiempos) == 0:
        print("No hubo trayectorias sincronizadas; no hay histograma de tiempos.")
        return
    fig, ax = plt.subplots(figsize=(8, 3))
    bins = range(min(tiempos), max(tiempos) + 2)
    ax.hist(tiempos, bins=bins, align="left", rwidth=0.85, color="steelblue", edgecolor="white")
    ax.set_xlabel("Tiempo de sincronización", fontsize=12)
    ax.set_ylabel("Frecuencia", fontsize=12)
    ax.set_title("Histograma de tiempos de sincronización", fontsize=13)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_mapa_regimenes(df):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(df["kappa"], df["rho_sync"], marker="o", label="$\\rho_{\\text{sync}}$",
            color="green", linewidth=2)
    ax.plot(df["kappa"], df["rho_fix"], marker="s", label="$\\rho_{\\text{fix}}$",
            color="orange", linewidth=2)
    ax.plot(df["kappa"], df["rho_cyc"], marker="^", label="$\\rho_{\\text{cyc}}$",
            color="purple", linewidth=2)
    ax.plot(df["kappa"], df["rho_transitorio"], marker="x", label="$\\rho_{\\text{trans}}$",
            color="gray", linewidth=1.5, linestyle="--")
    ax.set_xlabel("$\\kappa$", fontsize=13)
    ax.set_ylabel("Proporción", fontsize=12)
    ax.set_title("Mapa de regímenes dinámicos $\\rho(\\kappa)$", fontsize=13)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.show()


print("Funciones de visualización cargadas.")

Funciones de visualización cargadas.


## 6. Interfaz interactiva con widgets

In [7]:
# ── Widgets de parámetros ──────────────────────────────────────────────────
N_widget = widgets.IntSlider(
    value=4, min=2, max=7, step=1,
    description="N (vértices)",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="350px")
)
M_widget = widgets.IntSlider(
    value=6, min=2, max=12, step=1,
    description="M (fases)",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="350px")
)
kappa_widget = widgets.IntSlider(
    value=2, min=1, max=20, step=1,
    description="κ (umbral)",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="350px")
)
omega_widget = widgets.IntSlider(
    value=0, min=0, max=11, step=1,
    description="ω* (frecuencia)",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="350px")
)
epsilon_widget = widgets.IntSlider(
    value=0, min=0, max=6, step=1,
    description="ε (precisión)",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="350px")
)
T_widget = widgets.IntSlider(
    value=20, min=1, max=100, step=1,
    description="T (pasos)",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="350px")
)
grafo_widget = widgets.Dropdown(
    options=["Completo", "Camino", "Ciclo", "Estrella"],
    value="Completo",
    description="Grafo",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="350px")
)
modo_theta_widget = widgets.Dropdown(
    options=["Aleatoria", "Distribuida", "Manual ejemplo"],
    value="Aleatoria",
    description="theta0",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="350px")
)

# ── Botones ────────────────────────────────────────────────────────────────
boton_simular    = widgets.Button(description="▶ Simular trayectoria",
                                   button_style="success",
                                   layout=widgets.Layout(width="220px", height="40px"))
boton_exhaustivo = widgets.Button(description="⚙ Exploración exhaustiva",
                                   button_style="warning",
                                   layout=widgets.Layout(width="220px", height="40px"))
boton_barrido    = widgets.Button(description="📊 Barrido de kappa",
                                   button_style="info",
                                   layout=widgets.Layout(width="220px", height="40px"))

salida = widgets.Output()


# ── Construcción de theta0 ─────────────────────────────────────────────────
def construir_theta0(N, M, modo):
    if modo == "Manual ejemplo":
        # Usa valores fijos representativos
        base = [0, M // 3, (2 * M) // 3, M - 1, M // 2, 1]
        return np.array((base + [0] * N)[:N]) % M
    elif modo == "Distribuida":
        # Distribuye uniformemente las fases en Z_M
        return np.array([int(i * M / N) % M for i in range(N)])
    else:
        # Aleatoria
        return np.random.randint(0, M, size=N)


# ── Actualización automática de rangos ────────────────────────────────────
def actualizar_rangos(*args):
    try:
        N = N_widget.value
        M = M_widget.value
        omega_widget.max = M - 1
        if omega_widget.value > omega_widget.max:
            omega_widget.value = 0
        epsilon_widget.max = M // 2
        if epsilon_widget.value > epsilon_widget.max:
            epsilon_widget.value = 0
        tipo = grafo_widget.value
        modelo_temp = KuramotoDiscreto(N=N, M=M, kappa=1, omega=0,
                                       tipo_grafo=tipo, epsilon=0)
        degmax = max(dict(modelo_temp.G.degree()).values())
        kappa_max = max(1, degmax * (M // 2) + 1)
        kappa_widget.max = kappa_max
        if kappa_widget.value > kappa_widget.max:
            kappa_widget.value = min(2, kappa_max)
    except Exception:
        kappa_widget.max = 20


for w in [N_widget, M_widget, grafo_widget]:
    w.observe(actualizar_rangos, names="value")

actualizar_rangos()


# ── Callbacks de botones ───────────────────────────────────────────────────
def on_simular_clicked(b):
    with salida:
        clear_output(wait=True)
        N = N_widget.value
        M = M_widget.value
        kappa = kappa_widget.value
        omega = omega_widget.value
        epsilon = epsilon_widget.value
        T = T_widget.value
        tipo_grafo = grafo_widget.value
        modo = modo_theta_widget.value

        modelo = KuramotoDiscreto(N=N, M=M, kappa=kappa, omega=omega,
                                  tipo_grafo=tipo_grafo, epsilon=epsilon)
        theta0 = construir_theta0(N, M, modo)
        resultado = modelo.simular(theta0, T=T)
        clasif = modelo.clasificar_atractor(theta0, T_max=max(200, T))
        tabla = modelo.tabla_trayectoria(resultado)

        print("=" * 55)
        print("SIMULACIÓN INDIVIDUAL")
        print("=" * 55)
        print(f"Grafo:        {tipo_grafo} | N={N}, M={M}")
        print(f"Parámetros:   κ={kappa}, ω*={omega}, ε={epsilon}")
        print(f"theta0:       {tuple(theta0)}")
        print(f"Atractor:     {clasif['tipo']}")
        print(f"Tiempo:       {clasif['tiempo']}")
        print(f"Periodo:      {clasif['periodo']}")
        print(f"Estado final: {tuple(clasif['estado_final'])}")
        print("=" * 55)

        plot_heatmap(resultado, M)
        plot_order_parameter(resultado, clasif)

        traj = resultado["trayectoria"]
        n_snaps = min(3, len(traj))
        indices = sorted(set([0, len(traj) // 2, len(traj) - 1]))[:n_snaps]
        for ti in indices:
            plot_grafo(modelo, traj[ti], title=f"Snapshot t={ti}")

        print("\nTabla de trayectoria:")
        display(tabla)


def on_exhaustivo_clicked(b):
    with salida:
        clear_output(wait=True)
        N = N_widget.value
        M = M_widget.value
        kappa = kappa_widget.value
        omega = omega_widget.value
        epsilon = epsilon_widget.value
        tipo_grafo = grafo_widget.value
        total = M ** N

        print("=" * 55)
        print("EXPLORACIÓN EXHAUSTIVA")
        print("=" * 55)
        print(f"Configuraciones: M^N = {M}^{N} = {total:,}")
        print(f"Bell(N={N}) = {bell_number(N)}")

        if total > 20000:
            print("⚠ El espacio es muy grande. Reduce N ≤ 5 o M ≤ 6.")
            return

        print("Calculando... (puede tardar unos segundos)")
        resumen = exploracion_exhaustiva(N, M, kappa, omega, tipo_grafo, epsilon, T_max=150)

        df_resumen = pd.DataFrame([{
            "total": resumen["total_configuraciones"],
            "sync": resumen["sync"],
            "fix": resumen["fix"],
            "cyc": resumen["cyc"],
            "transitorio": resumen["transitorio"],
            "ρ_sync": round(resumen["rho_sync"], 4),
            "ρ_fix": round(resumen["rho_fix"], 4),
            "ρ_cyc": round(resumen["rho_cyc"], 4),
            "subredes_realizables": resumen["num_subredes_realizables"],
            "transiciones": resumen["num_transiciones"],
            "Bell_BN": resumen["Bell_BN"],
        }])

        display(df_resumen)
        plot_hist_tiempos(resumen["tiempos_sync"])


def on_barrido_clicked(b):
    with salida:
        clear_output(wait=True)
        N = N_widget.value
        M = M_widget.value
        omega = omega_widget.value
        epsilon = epsilon_widget.value
        tipo_grafo = grafo_widget.value
        total = M ** N

        print("=" * 55)
        print("BARRIDO DE KAPPA — MAPA DE REGÍMENES")
        print("=" * 55)
        print(f"Configuraciones por kappa: {M}^{N} = {total:,}")

        if total > 10000:
            print("⚠ El barrido puede ser muy pesado. Reduce N ≤ 4 o M ≤ 6.")
            return

        print("Calculando barrido... (puede tardar varios segundos)")
        df = barrido_kappa(N, M, omega, tipo_grafo, epsilon, T_max=150)
        display(df.round(4))
        plot_mapa_regimenes(df)

        if len(df) > 0:
            k_opt = int(df.loc[df["rho_sync"].idxmax(), "kappa"])
            rho_max = float(df["rho_sync"].max())
            print(f"\nκ de máxima sincronización: {k_opt}")
            print(f"ρ_sync máximo:              {rho_max:.4f}")


boton_simular.on_click(on_simular_clicked)
boton_exhaustivo.on_click(on_exhaustivo_clicked)
boton_barrido.on_click(on_barrido_clicked)


# ── Panel principal ────────────────────────────────────────────────────────
titulo = widgets.HTML("""
<div style='background:#f8f9fa; border-left:5px solid #2196F3; padding:14px 18px; margin-bottom:10px; border-radius:4px;'>
  <h2 style='margin:0 0 6px 0; color:#1a1a2e;'>Simulador de Sincronización Discreta tipo Kuramoto sobre Grafos</h2>
  <p style='margin:0; color:#444; font-size:13px;'>
    Modelo: <code>(F<sub>κ</sub>(θ))<sub>i</sub> = θ<sub>i</sub> + ω* + σ<sub>κ</sub>(S<sub>i</sub>(θ)) mod M</code>
    &nbsp;|&nbsp; S<sub>i</sub>(θ) = Σ<sub>j∈N(i)</sub> d<sup>signed</sup><sub>M</sub>(θ<sub>j</sub>, θ<sub>i</sub>)
  </p>
</div>
""")

separador = widgets.HTML("<hr style='margin:10px 0; border-color:#ddd;'>")

panel = widgets.VBox([
    titulo,
    widgets.HBox([
        widgets.VBox([N_widget, M_widget, kappa_widget, omega_widget]),
        widgets.VBox([epsilon_widget, T_widget, grafo_widget, modo_theta_widget]),
    ]),
    separador,
    widgets.HBox([boton_simular, boton_exhaustivo, boton_barrido],
                 layout=widgets.Layout(gap="12px")),
    salida
])

display(panel)